**Autor:** Alan Sastre  
**GitHub:** [github.com/alansastre/genai-agentes](https://github.com/alansastre/genai-agentes)

---

Crear **middleware custom** en LangChain consiste en interceptar puntos del ciclo del agente para aplicar reglas de negocio, control operativo y observabilidad sin tocar la lógica principal de herramientas ni prompts.

En el trabajo diario, este enfoque resulta útil cuando necesitas una regla **transversal** en todos los turnos, por ejemplo limitar mensajes, reintentar llamadas o registrar ejecución de herramientas.

## Cuándo usar middleware custom

Un middleware custom tiene sentido cuando la necesidad es **repetible** y no pertenece a una herramienta concreta. Si la misma validación aparece en varios agentes, es mejor moverla a un hook.

> La señal más clara es esta: "quiero aplicar la misma política en cada llamada al modelo o en cada tool call".

Casos típicos y sencillos:

* **Validación previa**: cortar una ejecución antes de llamar al modelo si se supera un límite de mensajes.
* **Reintento técnico**: repetir una llamada de modelo si falla por red o timeout.
* **Registro operativo**: trazar qué herramienta se ejecuta y con qué argumentos.

## Hooks esenciales

LangChain ofrece dos familias de **hooks** para middleware custom:

* **Node-style hooks**: se ejecutan en puntos concretos del flujo, como `before_model` o `after_model`.
* **Wrap-style hooks**: envuelven una llamada completa, como `wrap_model_call` y `wrap_tool_call`.

> Si solo quieres validar o anotar estado, usa node-style. Si necesitas controlar la ejecución del handler, usa wrap-style.

En ejecución real, el orden también importa: los hooks `before_*` van en orden de lista, mientras que `after_*` se ejecutan en orden inverso.

## Ejemplo 1: validación simple con `before_model`

Este ejemplo aplica una política **operativa** muy útil: si hay demasiados mensajes en el historial, devolver una respuesta corta y terminar el run con `jump_to: "end"`.

En uso real, cada **invoke** corresponde a un nuevo mensaje del usuario: pasas el estado actual (mensajes acumulados) más el nuevo `HumanMessage`. El middleware se ejecuta antes de llamar al modelo; si el número de mensajes supera el límite, corta y devuelve el aviso sin gastar llamada al LLM.

> Es una forma limpia de evitar bucles largos y controlar coste sin modificar herramientas.

In [1]:
from typing import Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentState, before_model
from langchain.messages import AIMessage, HumanMessage
from langgraph.runtime import Runtime

MAX_MESSAGES = 4

@before_model(can_jump_to=["end"])
def stop_if_too_long(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"[before_model] mensajes actuales: {len(state['messages'])}")
    # Regla transversal: cortar si el historial supera el límite.
    if len(state["messages"]) >= MAX_MESSAGES:
        return {
            "messages": [AIMessage(content="Límite de contexto alcanzado. Reinicia o resume la conversación.")],
            "jump_to": "end",
        }
    return None

agent = create_agent(
    model="ollama:gemma3:1b",
    tools=[],
    middleware=[stop_if_too_long],
)

# Simulamos una conversación: cada invoke es un nuevo turno del usuario (nuevo HumanMessage).
# El estado se acumula entre invocaciones como en una app real.
messages = [HumanMessage(content="Hola")]
result = agent.invoke({"messages": messages})
messages = result["messages"]
print("Turno 1 - Respuesta:", result["messages"][-1].content[:60] + "...")

messages.append(HumanMessage(content="¿Qué puedes hacer por mí?"))
result = agent.invoke({"messages": messages})
messages = result["messages"]
print("Turno 2 - Respuesta:", result["messages"][-1].content[:60] + "...")

# En el tercer turno el historial ya tiene 4 mensajes; al añadir el 5.º, el middleware corta.
messages.append(HumanMessage(content="Una pregunta más."))
result = agent.invoke({"messages": messages})
print("Turno 3 (límite alcanzado):", result["messages"][-1].content)

[before_model] mensajes actuales: 1
Turno 1 - Respuesta: ¡Hola! ¿En qué puedo ayudarte hoy? 😊
...
[before_model] mensajes actuales: 3
Turno 2 - Respuesta: ¡Puedo hacer muchas cosas! Aquí te dejo algunos ejemplos de ...
[before_model] mensajes actuales: 5
Turno 3 (límite alcanzado): Límite de contexto alcanzado. Reinicia o resume la conversación.


## Ejemplo 2: reintento alrededor del modelo con `wrap_model_call`

Aquí se implementa un reintento **mínimo** que cubre errores transitorios. El middleware llama al handler una vez, y si falla, reintenta hasta un máximo definido.

> Este patrón evita duplicar bloques `try/except` en cada parte de tu aplicación.

In [2]:
from typing import Callable
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from langchain.messages import HumanMessage

retry_state = {"simulated_failure_done": False}

@wrap_model_call
def retry_model_call(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    max_retries = 3
    for attempt in range(max_retries):
        try:
            # Fuerza un fallo en el primer intento para ver el reintento.
            if not retry_state["simulated_failure_done"]:
                retry_state["simulated_failure_done"] = True
                raise TimeoutError("Fallo simulado de red")
            print(f"[wrap_model_call] intento {attempt + 1}: llamada real al modelo")
            return handler(request)
        except Exception as exc:
            # Solo relanza en el último intento.
            if attempt == max_retries - 1:
                raise
            print(f"Reintento {attempt + 1}/{max_retries} por error técnico: {exc}")

agent = create_agent(
    model="ollama:gemma3:1b",
    tools=[],
    middleware=[retry_model_call],
)

result = agent.invoke({"messages": [HumanMessage(content="Explica en una frase qué es una API REST.")]})
print("Respuesta final:", result["messages"][-1].content)

Reintento 1/3 por error técnico: Fallo simulado de red
[wrap_model_call] intento 2: llamada real al modelo
Respuesta final: Una API REST (Representational State Transfer) es un conjunto de reglas y especificaciones que permite a diferentes sistemas comunicarse e intercambiar información a través de una interfaz basada en recursos.


## Ejemplo 3: control de herramientas con `wrap_tool_call`

Este middleware añade trazabilidad **diaria**: imprime nombre y argumentos de cada herramienta, y registra si la ejecución fue correcta o fallida.

> Cuando un agente llama varias tools, este hook ahorra tiempo de depuración desde el primer día.

In [4]:
import uuid
from typing import Callable
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import HumanMessage, ToolMessage
from langchain.tools import tool
from langchain.tools.tool_node import ToolCallRequest
from langgraph.types import Command

# Valor dinámico para dificultar respuestas inventadas.
SECRET_CODE = str(uuid.uuid4())[:8]

@tool
def obtener_codigo_secreto() -> str:
    """Devuelve un código secreto temporal."""
    return SECRET_CODE

@wrap_tool_call
def monitor_tool_call(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage | Command],
) -> ToolMessage | Command:
    tool_name = request.tool_call["name"]
    tool_args = request.tool_call["args"]
    print(f"[tool] Ejecutando {tool_name} con args={tool_args}")
    try:
        response = handler(request)
        print(f"[tool] {tool_name} finalizó correctamente")
        return response
    except Exception as exc:
        print(f"[tool] {tool_name} falló: {exc}")
        raise

agent = create_agent(
    model="gpt-5-nano",
    tools=[obtener_codigo_secreto],
    middleware=[monitor_tool_call],
)

result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "Debes usar la herramienta para recuperar el código secreto "
                    "y responder solo con ese valor."
                )
            ),
        ]
    }
)
tool_messages = [m for m in result["messages"] if m.__class__.__name__ == "ToolMessage"]
print("ToolMessage detectados:", len(tool_messages))
print("Código esperado:", SECRET_CODE)
print("Respuesta final:", result["messages"][-1].content)

[tool] Ejecutando obtener_codigo_secreto con args={}
[tool] obtener_codigo_secreto finalizó correctamente
ToolMessage detectados: 1
Código esperado: 855a1e39
Respuesta final: 855a1e39


## Clase o decorador: criterio práctico

Para tareas cortas, el formato con **decoradores** suele ser suficiente y mantiene el código ligero. Para middleware con configuración, varios hooks o variantes sync y async, conviene usar una clase que herede de `AgentMiddleware`.

> Regla simple: empieza con decorador y pasa a clase cuando necesites estado interno o más de un hook cohesionado.

Checklist de diseño para producción:

* **Responsabilidad única**: cada middleware debe resolver una sola preocupación técnica.
* **Errores controlados**: define si reintentas, bloqueas o relanzas, pero evita silencios.
* **Orden explícito**: coloca primero middleware críticos de seguridad o validación.
* **Pruebas unitarias**: valida cada middleware de forma aislada antes de combinarlo.
* **Estado documentado**: si amplías `AgentState`, deja claro qué campos añade el middleware.